# SALM Fine-tuning 03 · Answer the question, or admit you can't: spoken QA (HeySQuAD)

> **Task** spoken question + text passage → short answer or "unanswerable" · **Data** HeySQuAD (human voices)
> **Metric** SQuAD-v2 F1 ↑ · **Result** 43.30 → **84.37** F1 (+94.9%) · **Time** ~1.5 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: a chatty reader that never says "I don't know"

Someone asks a question aloud, e.g. *"What did Microsoft announce that it would rename OneDrive to?"* The model gets the
text passage in the prompt and the question as audio. It must answer with a span copied from the passage, or say
**unanswerable** when the passage doesn't contain one.

The base model reads well but rarely abstains. With the dataset's own prompt it often answers with whole sentences,
even when there is no answer:

```
gold   (unanswerable)
BASE   The kilogram-force leads to an alternate, but rarely used unit of mass: the metric slug.
```

The HeySQuAD human test set is **76% unanswerable**, while the training pool is 28%, so abstaining well matters. The
goal is a model that answers with short spans when it can and abstains when it can't.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data, and a passage-disjoint dev set

The passage goes into the *prompt*, the question is the *audio*, and the target is the first gold span (or
`unanswerable`). SQuAD has several questions per passage, so dev must not share passages with train; otherwise it
overestimates accuracy.

In [ ]:
d = M / "heysquad"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py heysquad --train-shards 8", log="prep_heysquad.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
import hashlib
P = [json.loads(l) for l in open(d / "train_pool.json")]
in_dev = lambda r: int(hashlib.md5(r["passage"].encode()).hexdigest(), 16) % 18 == 0
(d / "dev.json").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in P if in_dev(r)))
(d / "train.json").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in P if not in_dev(r)))
for s in ("train", "dev", "validation"):
    rows = [json.loads(l) for l in open(d / f"{s}.json")]
    print(f"{s:10s} {len(rows):6d} questions, {sum(r['is_impossible'] for r in rows) / len(rows):.0%} unanswerable")

## 2. Baseline: good answers, too few abstentions

In [ ]:
TASK = "--task squad --max-tokens 64 --max-model-len 8192"
P2 = ("Passage: {passage}\\n\\nListen to the question and answer it with a short phrase copied from the passage. "
      "If the passage does not answer it, reply unanswerable. Reply with the answer only.")
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt-template {q(P2)}")):   # p1 = the manifest's own context
    for s, man in (("dev", "dev"), ("test", "validation")):
        r = evaluate(CKPT, d / f"{man}.json", f"sqa_base_{p}_{s}", TASK, extra)
        base[p, s] = r
    print(f"{p}  dev F1 {base[p, 'dev']['score']:5.2f}   test F1 {base[p, 'test']['score']:5.2f}   "
          f"(test has-answer {base[p, 'test']['by_type']['has_ans']['f1']:.1f} / no-answer {base[p, 'test']['by_type']['no_ans']['f1']:.1f})")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"]["score"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]["score"]  # its test score: the baseline fine-tuning is compared with
ORACLE = max(base[p, "test"]["score"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f} F1")

p2 (passage first, "answer only") wins on dev by 10.5 points. The baseline is its test score, **43.30** F1, which is
also the test-oracle. It answers well (82.0 has-answer F1 on test) but abstains on only about a third of the
unanswerable questions (31.0 no-answer F1). Dev (27% unanswerable) forgives that; test (76%) punishes it. The
fine-tune trains with p2.

In [ ]:
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "validation"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(P2)} {pairs}")

## 3. Config: size by tokens, not hours

Each example carries a ~200-token passage but only ~80 audio tokens. The text dominates the token budget, so
`batch_tokens` is sized on the total.

In [ ]:
sh(f"{PYTHON} {FT}/epoch_batches.py {d}/train_ft.json 48000 --tokenizer {CKPT}")
CFG = WORK / "configs" / "sqa.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'sqa'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name sqa_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 30 --max-steps 300 "
   f"--batch-tokens 48000 --limit-train-batches 75 --limit-val-batches 25 --save-top-k -1")

## 4. Train (~3.4 s/step, ~20 min)

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 150 225 300")

## 5. Select on dev: watch the trade-off

In [ ]:
dev = {}
for step in (150, 225, 300):
    out = EXPORT / f"sqa_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'sqa'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_sqa_{step}.log")
    r = evaluate(out, d / "dev_ft.json", f"sqa_s{step}_dev", TASK)
    dev[step] = r["score"]
    print(f"step {step}: dev F1 {r['score']:.2f}  (has-answer {r['by_type']['has_ans']['f1']:.1f} / no-answer {r['by_type']['no_ans']['f1']:.1f})")
BEST = max(dev, key=dev.get)
print("selected step", BEST)

Expect the checkpoints to trade *answering* against *abstaining*: step 150 → 73.20 dev F1 (has-answer 71.7,
no-answer 77.3), step 225 → 74.56 (has-answer 70.2, no-answer 86.3), step 300 → 74.22 (has-answer 76.4, no-answer
68.2). Step 225 is selected by overall dev F1. Don't switch to a rule that favors abstention because you know the test
set does.

## 6. Test, once

In [ ]:
MODEL = EXPORT / f"sqa_s{BEST}"
ft = evaluate(MODEL, d / "validation_ft.json", "sqa_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "sqa_librispeech", "--task asr --normalizer english")
print(f"test F1: base {BASELINE:.2f} → fine-tuned {ft['score']:.2f} ({(ft['score'] - BASELINE) / BASELINE:+.1%})  EM {ft['exact_match']:.2f}")
print(f"has-answer F1 {ft['by_type']['has_ans']['f1']:.1f}   no-answer F1 {ft['by_type']['no_ans']['f1']:.1f}   LibriSpeech WER {100 * guard['score']:.2f}%")
samples("sqa_TEST", n=4, keys=("question", "answers", "pred_text"))
result("03 baseline test F1", BASELINE, 43.30, 0.5)
result("03 fine-tuned test F1", ft["score"], 84.37, 2.0)

Expected results:

| | test F1 | EM | has-answer F1 (1,002) | no-answer F1 (3,156) | LibriSpeech WER |
|---|---:|---:|---:|---:|---:|
| base, prompt chosen on dev (p2) | 43.30 | 40.14 | **82.0** | 31.0 | 1.54% |
| **fine-tuned** | **84.37** | **83.21** | 77.4 | **86.6** | 1.92% |

**+94.9% relative.** The gain comes from abstaining: no-answer F1 rises from 31.0 to 86.6, while answerable questions
drop from 82.0 to 77.4 F1. On a test set that is 76% unanswerable the trade pays off, but it is a trade, which is why
the split is reported.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="sqa")

## Takeaways

1. **Know your test set's priors.** 76% unanswerable changes which behavior matters.
2. **Choose the baseline prompt on dev**, the same way the checkpoint is chosen, and print the test-oracle beside it.
3. **Report the split** (has-answer / no-answer), so nobody wins by abstaining alone.